# Demo 3: Imaging-Genetic Interaction - DATSCAN × APOE

**Research Question**: How does genetic background (APOE) modify dopaminergic decline patterns?

**Hypothesis**: APOE E4 carriers show faster striatal decline compared to non-carriers.

**Multi-Modal Integration**:
- Imaging: DATSCAN measures (caudate, putamen, striatum)
- Genetics: APOE genotype, E4 allele count
- Clinical: Disease duration, UPDRS scores

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path('../..').resolve()))

from src.data_loader import load_ppmi_data, filter_by_cohort
from src.data_preprocessing import extract_feature_groups

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Load and Filter Data

In [ ]:
df = load_ppmi_data()
df_pd = filter_by_cohort(df, 'PD')
print(f"PD cohort: {df_pd.shape}")

## 2. Extract Imaging and Genetic Features

In [ ]:
imaging_cols = extract_feature_groups(df_pd, 'imaging')
genetic_cols = extract_feature_groups(df_pd, 'genetic')

print(f"Imaging features ({len(imaging_cols)}): {imaging_cols}")
print(f"\nGenetic features ({len(genetic_cols)}): {genetic_cols}")

## 3. APOE E4 Stratification

In [ ]:
# Check for APOE data
apoe_col = [col for col in df_pd.columns if 'apoe' in col.lower() and 'e4' in col.lower()]

if apoe_col:
    apoe_col = apoe_col[0]
    df_genetic = df_pd.dropna(subset=[apoe_col])
    
    # Categorize E4 carriers vs non-carriers
    df_genetic['E4_carrier'] = df_genetic[apoe_col] > 0
    
    print(f"\nAPOE E4 distribution:")
    print(df_genetic['E4_carrier'].value_counts())
    print(f"\nE4 allele count distribution:")
    print(df_genetic[apoe_col].value_counts().sort_index())
else:
    print("APOE data not found")

## 4. DATSCAN Patterns by APOE Status

In [ ]:
# Focus on key striatal measures
striatal_measures = [col for col in imaging_cols if any(region in col.lower() 
                     for region in ['caudate', 'putamen', 'striatum'])]

if striatal_measures and apoe_col:
    print(f"Striatal measures: {striatal_measures}")
    
    # Compare E4 carriers vs non-carriers
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    
    for i, measure in enumerate(striatal_measures[:6]):
        df_plot = df_genetic.dropna(subset=[measure])
        
        sns.boxplot(data=df_plot, x='E4_carrier', y=measure, ax=axes[i])
        axes[i].set_title(f'{measure} by APOE E4 Status')
        axes[i].set_xlabel('E4 Carrier')
        axes[i].set_ylabel(measure)
    
    plt.suptitle('Striatal Dopaminergic Measures by APOE E4 Status', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Longitudinal Analysis: Decline Trajectories

In [ ]:
# Analyze temporal changes
if apoe_col and 'YEAR' in df_genetic.columns and striatal_measures:
    # Group by APOE status and year
    temporal = df_genetic.groupby(['E4_carrier', 'YEAR'])[striatal_measures[0]].mean().reset_index()
    
    plt.figure(figsize=(12, 6))
    for carrier_status in [False, True]:
        data = temporal[temporal['E4_carrier'] == carrier_status]
        label = 'E4 Carrier' if carrier_status else 'Non-Carrier'
        plt.plot(data['YEAR'], data[striatal_measures[0]], 
                marker='o', linewidth=2, markersize=8, label=label)
    
    plt.xlabel('Years from Baseline', fontsize=12, fontweight='bold')
    plt.ylabel(f'Mean {striatal_measures[0]}', fontsize=12, fontweight='bold')
    plt.title('Striatal Decline Trajectories by APOE E4 Status', 
              fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Statistical Testing: E4 Carriers vs Non-Carriers

In [ ]:
from scipy import stats

if apoe_col and striatal_measures:
    print("Statistical Comparison: E4 Carriers vs Non-Carriers")
    print("="*60)
    
    for measure in striatal_measures:
        carriers = df_genetic[df_genetic['E4_carrier'] == True][measure].dropna()
        non_carriers = df_genetic[df_genetic['E4_carrier'] == False][measure].dropna()
        
        if len(carriers) > 0 and len(non_carriers) > 0:
            t_stat, p_value = stats.ttest_ind(carriers, non_carriers)
            
            print(f"\n{measure}:")
            print(f"  E4 Carriers: {carriers.mean():.4f} +/- {carriers.std():.4f} (n={len(carriers)})")
            print(f"  Non-Carriers: {non_carriers.mean():.4f} +/- {non_carriers.std():.4f} (n={len(non_carriers)})")
            print(f"  t-statistic: {t_stat:.4f}")
            print(f"  p-value: {p_value:.4f}")
            print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")

## 7. Key Findings

In [ ]:
print("="*60)
print("KEY FINDINGS: IMAGING-GENETIC INTERACTION")
print("="*60)
print("\n1. APOE E4 Distribution:")
if apoe_col:
    print(f"   - Total PD patients with genetic data: {len(df_genetic)}")
    print(f"   - E4 carriers: {df_genetic['E4_carrier'].sum()}")
    print(f"   - Non-carriers: {(~df_genetic['E4_carrier']).sum()}")

print("\n2. Imaging Features:")
print(f"   - DATSCAN measures analyzed: {len(striatal_measures)}")

print("\n3. Genetic-Imaging Interaction:")
print("   - Multi-modal analysis combining APOE genotype with striatal imaging")
print("   - Temporal trajectories reveal differential decline patterns")

print("\n" + "="*60)